# 1.06 Data Collection (API)

## What students learn in this lab
- What an API is (a service that returns data, often as **JSON**)
- How to send an HTTP **GET** request with `requests`
- How to validate the response (status codes + errors)
- How to convert JSON → **pandas DataFrame**
- A small “real workflow”: clean/select columns and save to CSV

We will use a free practice API: **JSONPlaceholder** (no API key needed).

In [ ]:
import json
import requests
import pandas as pd

# API endpoint (JSONPlaceholder is great for learning)
url = "https://jsonplaceholder.typicode.com/users"

# Good practice: add a timeout + a User-Agent
headers = {"User-Agent": "ml-course-student/1.0"}

# Send GET request (HTTP)
try:
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()  # raises an error for 4xx/5xx
except requests.RequestException as e:
    raise SystemExit(f"Request failed: {e}")

response  # shows basic info in notebooks

In [ ]:
# Check status + content type
print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

# Tip: 200 means OK. 404 means not found. 401/403 means auth issues. 500 means server error.

Status code: 200


In [ ]:
# Parse JSON (Python objects)
data = response.json()  # list[dict] for this endpoint
print("Type:", type(data))
print("Number of records:", len(data))

In [ ]:
# Preview 1 record (pretty printed)
print(json.dumps(data[0], indent=2)[:800])  # limit output length

[{'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'}},
 {'id': 2,
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'email': 'Shanna@melissa.tv',
  'address': {'street': 'Victor Plains',
   'suite': 'Suite 879',
   'city': 'Wisokyburgh',
   'zipcode': '90566-7771',
   'geo': {'lat': '-43.9509', 'lng': '-34.4618'}},
  'phone': '010-692-6593 x09125',
  'website': 'anastasia.net',
  'company': {'name': 'Deckow-Crist',
   'catchPhrase': 'Proactive didactic contingency',
   'bs': 'synergize scalable supply-chains'}},
 {'id': 3,
  'name': 'Clementine Bauch',
  'username': 'Samantha

In [ ]:
# JSON -> DataFrame
# json_normalize flattens nested objects like address.city, company.name
df_users = pd.json_normalize(data)

print("Columns:")
print(df_users.columns.tolist())

df_users.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [ ]:
# Keep only a few useful columns (clean student-friendly table)
cols = [
    "id", "name", "username", "email",
    "address.city", "company.name",
]
df_clean = df_users[cols].copy()
df_clean.rename(columns={"address.city": "city", "company.name": "company"}, inplace=True)

df_clean.head()

In [ ]:
# Save for later use (CSV)
output_path = "output/1.06_users_from_api.csv"
df_clean.to_csv(output_path, index=False)
print("Saved to:", output_path)

### Extra (optional): join two API endpoints
We can request `posts` and merge them with `users` using the user id (this is common in real data projects).

In [ ]:
posts_url = "https://jsonplaceholder.typicode.com/posts"
posts = requests.get(posts_url, headers=headers, timeout=20).json()
df_posts = pd.json_normalize(posts)[["userId", "id", "title"]]
df_posts.rename(columns={"id": "post_id"}, inplace=True)

# Merge users with posts
df_joined = df_clean.merge(df_posts, left_on="id", right_on="userId", how="left")
df_joined.head()